# Amica: train the Scan before you ride models (Kaggle)

This notebook builds the on-device models for the vehicle check. The features are off in the app until you run it:

| Model | App file | Turns on |
|---|---|---|
| **Plate finder**, trained here on Sri Lankan plates | `plate_detector.tflite` + `.txt` | `FeatureFlags.plateFinder` |
| **Vehicle type**, the ready-made COCO YOLOv8n, exported | `vehicle_detector.tflite` + `.txt` | `FeatureFlags.vehicleCheck` |
| **Sri Lanka vehicle types with three-wheelers** (optional, needs your labelled photos) | `vehicle_detector_lk.tflite` + `.txt` | `FeatureFlags.vehicleCheck` |

## Kaggle setup (once)

In the notebook editor's right-hand panel, under **Session options**:

1. **Accelerator:** choose **GPU T4 x2**. Avoid the P100, because newer PyTorch builds no longer support it.
2. **Internet:** turn it **On**. The notebook installs packages and downloads the YOLO weights. Kaggle requires a phone-verified account for this.
3. **Add Input** (the *+ Add Input* button):
   * **Training scripts:** upload `amica-ai-core/plate_ocr/training` as a private Kaggle Dataset (for example `amica-training`) and add it here. You can skip this if you set `REPO_URL` in section 2.
   * **Plate data:** add a Kaggle dataset of Sri Lankan plates (search "Sri Lanka number plate"). YOLO `.txt` labels and Pascal VOC `.xml` annotations both work. You can also use Roboflow in section 3.
4. **Roboflow key** (only if you use Roboflow): *Add-ons → Secrets → Add secret*, with the label `ROBOFLOW_API_KEY`, then tick it for this notebook.

Then choose **Run All**. The finished models land in `/kaggle/working/amica_models/`, with a download link at the end. **Save Version** keeps them in the notebook's *Output* tab.

**Data licences:** the Roboflow "Srilankan Number Plates" dataset is CC BY 4.0. Check the licence of any Kaggle dataset you add, and credit your sources in the report and in the app.

## 1. Settings

Change these values, then run the cell. The defaults work with **Run All** once the inputs above are added.

In [ ]:
import os

# Where the plate dataset comes from:
#   "kaggle"   - a dataset added with "+ Add Input" (found under /kaggle/input)
#   "roboflow" - download with the ROBOFLOW_API_KEY secret
#   "folder"   - any local YOLO / VOC folder (DATA_FOLDER)
DATA_SOURCE = os.environ.get("AMICA_DATA_SOURCE", "kaggle")
KAGGLE_DATASET = ""            # e.g. "sri-lanka-number-plates"; "" = auto-detect
ROBOFLOW_WORKSPACE = ""        # from Roboflow: Download Dataset -> YOLOv8 -> show download code
ROBOFLOW_PROJECT = ""
ROBOFLOW_VERSION = 1
DATA_FOLDER = os.environ.get("AMICA_DATA_DIR", "")

# Extra datasets to merge in (paths), e.g. your group's own labelled photos.
EXTRA_FOLDERS = []

# Training. 100 epochs take about 10 minutes on a T4 for ~250 images.
EPOCHS = int(os.environ.get("AMICA_EPOCHS", 100))
PLATE_IMGSZ = int(os.environ.get("AMICA_IMGSZ", 416))   # plates are small: 416 px input
BATCH = 16
BASE_WEIGHTS = os.environ.get("AMICA_BASE_WEIGHTS", "yolov8n.pt")

# Optional sections.
EXPORT_COCO_VEHICLE_MODEL = True   # vehicle_detector.tflite (no training needed)
TRAIN_THREE_WHEELER_MODEL = False  # vehicle_detector_lk.tflite, see section 8b
VEHICLE_KAGGLE_DATASET = "vehicle-dataset-for-yolo"  # Kaggle input to train it on
VEHICLE_DATA_FOLDER = ""           # or a local YOLO/VOC folder instead
VEHICLE_EPOCHS = int(os.environ.get("AMICA_VEHICLE_EPOCHS", 80))
RUN_EVALUATION = False             # needs your own test photos, see section 9
EVAL_IMAGES = ""
EVAL_CSV = ""                      # image,plate,vehicle_type,colour,condition

ON_KAGGLE = os.path.isdir("/kaggle/working")
# Saved with the notebook version (Output tab):
OUTPUT_DIR = "/kaggle/working/amica_models" if ON_KAGGLE else os.path.abspath("amica_models")
# Scratch (datasets, training runs); not saved, so versions stay small:
WORK_DIR = "/tmp/amica_work" if ON_KAGGLE else os.path.abspath("work")

## 2. Install and find the training scripts

This cell looks for `train.py` in an attached Kaggle input first, then next to this notebook when you run it inside the repo, and finally clones `REPO_URL` if you set it.

Only the packages Kaggle doesn't already have are installed. **Kaggle's core libraries stay at their current versions:** numpy, protobuf, TensorFlow, PyTorch, pandas, OpenCV and a few others. pip picks versions of the new packages that fit around them. Upgrading protobuf or numpy would break TensorFlow and the export step. pip's output is shown only if an install fails, because Kaggle's image has unrelated dependency conflicts (google-colab, bigframes, …) that pip would otherwise list every time.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = ""  # e.g. "https://github.com/<you>/amica-ai-core.git" (optional)

def find_training_dir():
    roots = [Path("/kaggle/input"), Path("../plate_ocr/training"), Path("plate_ocr/training"),
             Path("amica-ai-core/plate_ocr/training"), Path("training")]
    for root in roots:
        if (root / "train.py").exists() and (root / "export_tflite.py").exists():
            return root.resolve()
        if root.name == "input" and root.exists():
            for hit in sorted(root.rglob("export_tflite.py")):
                return hit.parent.resolve()
    return None

TRAINING_DIR = find_training_dir()
if TRAINING_DIR is None and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/tmp/amica-ai-core"], check=True)
    TRAINING_DIR = Path("/tmp/amica-ai-core/plate_ocr/training")
assert TRAINING_DIR and (TRAINING_DIR / "train.py").exists(), (
    "Training scripts not found: attach the plate_ocr/training folder as a Kaggle "
    "dataset (+ Add Input) or set REPO_URL.")
print("Training scripts:", TRAINING_DIR)

from importlib.metadata import distributions, version, PackageNotFoundError

# Kaggle's image already has numpy, protobuf, TensorFlow, PyTorch and
# friends. Installing new packages must not upgrade them: a newer protobuf
# or numpy breaks TensorFlow and the export step. So these core libraries
# are pinned at their current versions, and pip picks versions of the new
# packages that fit around them. (Pinning *every* package fails: Kaggle's
# image already has conflicts between unrelated packages.)
PROTECTED = [
    "numpy", "protobuf", "ml-dtypes", "tensorflow", "tensorflow-cpu", "tensorboard",
    "keras", "tf-keras", "torch", "torchvision", "torchaudio", "jax", "jaxlib",
    "pandas", "scipy", "pillow", "matplotlib", "h5py", "grpcio",
    "opencv-python", "opencv-python-headless", "opencv-contrib-python",
]
CONSTRAINTS = WORK_DIR + "_constraints.txt"
os.makedirs(WORK_DIR, exist_ok=True)
with open(CONSTRAINTS, "w") as f:
    for name in PROTECTED:
        try:
            f.write(f"{name}=={version(name)}\n")
        except PackageNotFoundError:
            pass

def pip(*packages, pin=True):
    # Install quietly. Kaggle's image has pre-existing dependency conflicts
    # (google-colab, bigframes, ...) that pip lists after every install; they
    # are unrelated to this notebook, so pip's output is shown only on failure.
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    if pin:
        cmd += ["-c", CONSTRAINTS]
    result = subprocess.run(cmd + list(packages), capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError(f"pip install {' '.join(packages)} failed (see above)")

if ON_KAGGLE:
    # roboflow is installed later, only if the Roboflow source is used.
    # onnxruntime is listed so Ultralytics doesn't auto-install it mid-export.
    pip("ultralytics", "onnx>=1.16", "onnxslim", "onnxruntime", "onnx2tf",
        "sng4onnx", "onnx_graphsurgeon", "psutil")
    # Read TensorFlow's version without importing it: loading TensorFlow and
    # PyTorch into the same process before training can crash the kernel.
    try:
        tf_version = version("tensorflow")
    except PackageNotFoundError:
        pip("tensorflow-cpu", pin=False); tf_version = version("tensorflow-cpu")
    major_minor = ".".join(tf_version.split(".")[:2])
    # onnx2tf needs tf_keras matching TensorFlow's version. Its only
    # dependency is TensorFlow itself, which is already here, so install it
    # without re-resolving TensorFlow's dependency tree.
    try:
        have = version("tf_keras")
    except PackageNotFoundError:
        have = ""
    if ".".join(have.split(".")[:2]) != major_minor:
        pip(f"tf_keras~={major_minor}.0", "--no-deps", pin=False)
    # onnx2tf runs its own check with Google's LiteRT interpreter
    # (ai-edge-litert), which Kaggle doesn't ship.
    try:
        pip("ai-edge-litert")
    except RuntimeError:
        # Its pins clash with Kaggle's image: install it without changing
        # the preinstalled packages, plus the two small deps Kaggle lacks.
        pip("ai-edge-litert", "--no-deps", pin=False)
        pip("backports.strenum", "flatbuffers")

    # Fail now, not after training, if the exporter can't load.
    check = subprocess.run([sys.executable, "-c", "import onnx2tf, ai_edge_litert, tf_keras"],
                           capture_output=True, text=True)
    if check.returncode != 0:
        print(check.stderr[-3000:])
        raise RuntimeError("The TFLite exporter (onnx2tf) can't load - see the error above")
    print("TFLite exporter ready")

    for package in ["numpy", "protobuf", "torch", "ultralytics", "onnx", "onnx2tf",
                    "ai-edge-litert", "tf_keras"]:
        try:
            print(f"{package:12} {version(package)}")
        except PackageNotFoundError:
            print(f"{package:12} not installed")
    print(f"{'tensorflow':12} {tf_version}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
WORK = Path(WORK_DIR); WORK.mkdir(parents=True, exist_ok=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "none - turn on GPU T4 x2 in Session options (CPU training is slow)")
DEVICE = os.environ.get("AMICA_DEVICE", "0" if torch.cuda.is_available() else "cpu")

def run(script, *args):
    # Run a training script and stream its output.
    cmd = [sys.executable, str(TRAINING_DIR / script), *map(str, args)]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=WORK)

## 3. Get the plate dataset

For a **Kaggle input**, this cell finds the attached folder that holds labelled images. It skips the folder of training scripts. Both YOLO `.txt` labels and Pascal VOC `.xml` annotations are read. `.txt` files that hold only the plate text, not boxes, are ignored.

All class names, such as "number plate" or "License_Plate", become one class, `plate`. If the dataset has no test split, one is made. Kaggle inputs are read-only, so the prepared copy goes to a scratch folder.

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def looks_labelled(folder):
    # True when a folder holds images plus YOLO labels or VOC annotations.
    images = sum(1 for p in folder.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
    labels = sum(1 for p in folder.rglob("*") if p.suffix.lower() in {".txt", ".xml"})
    return images > 0 and labels > 0

inputs = Path("/kaggle/input")

def dataset_roots():
    # /kaggle/input/<slug>, or the newer /kaggle/input/datasets/<owner>/<slug>
    for p in sorted(inputs.rglob("*")) if inputs.exists() else []:
        rel = p.relative_to(inputs).parts
        if p.is_dir() and (len(rel) == 1 and rel[0] not in ("datasets", "competitions")
                           or len(rel) == 3 and rel[0] in ("datasets", "competitions")):
            yield p


args = []
if DATA_SOURCE == "kaggle":
    def holds_scripts(p):
        return TRAINING_DIR == p or p in TRAINING_DIR.parents

    candidates = [p for p in dataset_roots()
                  if (not KAGGLE_DATASET or KAGGLE_DATASET in str(p))
                  and not holds_scripts(p)
                  and not (VEHICLE_KAGGLE_DATASET and VEHICLE_KAGGLE_DATASET in str(p))
                  and looks_labelled(p)]
    # Prefer a dataset with "plate" in its name when several are attached.
    candidates.sort(key=lambda p: "plate" not in p.name.lower())
    assert candidates, "No labelled plate dataset under /kaggle/input - add one with + Add Input"
    source = candidates[0]
    print("Using Kaggle dataset:", source)
    if len(candidates) > 1:
        print("Other candidates (set KAGGLE_DATASET to pick one):", [str(c) for c in candidates[1:]])
    args = ["--source", source]
elif DATA_SOURCE == "roboflow":
    assert ROBOFLOW_WORKSPACE and ROBOFLOW_PROJECT, "Fill ROBOFLOW_WORKSPACE and ROBOFLOW_PROJECT in section 1"
    key = os.environ.get("ROBOFLOW_API_KEY")
    if not key and ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    if not key:
        from getpass import getpass
        key = getpass("Roboflow API key: ")
    if ON_KAGGLE:
        try:
            pip("roboflow")
        except RuntimeError:
            # roboflow pins a few exact versions; if they clash with Kaggle's
            # image, install it without touching the preinstalled packages.
            pip("roboflow", "--no-deps", pin=False)
            pip("requests-toolbelt", "python-dotenv", "filetype", "pillow-heif")
    args = ["--roboflow-key", key, "--workspace", ROBOFLOW_WORKSPACE,
            "--project", ROBOFLOW_PROJECT, "--version", ROBOFLOW_VERSION]
else:
    assert DATA_FOLDER, "Set DATA_FOLDER in section 1"
    args = ["--source", Path(DATA_FOLDER).resolve()]
for extra in EXTRA_FOLDERS:
    args += ["--extra", Path(extra).resolve()]

run("prepare_dataset.py", "--task", "plates", "--out", WORK / "datasets/plates", *args)
PLATE_DATA = WORK / "datasets/plates/data.yaml"

## 4. Look at the data

Check that the pink boxes sit on the plates. If they don't, the annotation format was read wrongly.

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
%matplotlib inline

def show_labelled(split_dir, n=6, title=""):
    images = sorted((split_dir / "images").iterdir())
    random.Random(0).shuffle(images)
    n = min(n, len(images))
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), squeeze=False)
    for ax, path in zip(axes[0], images):
        im = Image.open(path).convert("RGB"); d = ImageDraw.Draw(im); w, h = im.size
        for line in (split_dir / "labels" / f"{path.stem}.txt").read_text().splitlines():
            _, cx, cy, bw, bh = map(float, line.split()[:5])
            d.rectangle([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h],
                        outline=(255, 0, 180), width=max(2, w // 200))
        ax.imshow(im); ax.axis("off")
    fig.suptitle(title); plt.show()

root = PLATE_DATA.parent
for split in ["train", "valid", "test"]:
    print(split, len(list((root / split / "images").iterdir())), "images")
show_labelled(root / "train", title="Training images with plate boxes")

## 5. Train the plate finder

Training starts from COCO-pretrained YOLOv8n weights, which is what makes a small dataset workable. Horizontal flips are off so plate text is never mirrored, and colour and angle augmentation stands in for night, street lights and angled shots. At the end, the script prints the **test mAP50**; put it in the report.

In [ ]:
run("train.py", "--data", PLATE_DATA, "--name", "plate_detector", "--imgsz", PLATE_IMGSZ,
    "--epochs", EPOCHS, "--batch", BATCH, "--device", DEVICE, "--model", BASE_WEIGHTS)
PLATE_WEIGHTS = WORK / "runs/plate_detector/weights/best.pt"
assert PLATE_WEIGHTS.exists(), PLATE_WEIGHTS

## 6. How well does it work?

This section shows the training curves, then runs the trained finder on test images it never saw during training. The curves and the best `.pt` weights are also copied to the output folder, for the report.

In [ ]:
import shutil
from IPython.display import Image as ShowImage, display
from ultralytics import YOLO

run_dir = PLATE_WEIGHTS.parent.parent
for name in ["results.png", "confusion_matrix.png", "PR_curve.png", "BoxPR_curve.png"]:
    if (run_dir / name).exists():
        shutil.copy(run_dir / name, Path(OUTPUT_DIR) / f"plate_{name}")
if (run_dir / "results.png").exists():
    display(ShowImage(filename=str(run_dir / "results.png"), width=900))
shutil.copy(PLATE_WEIGHTS, Path(OUTPUT_DIR) / "plate_detector_best.pt")

plate_model = YOLO(PLATE_WEIGHTS)
metrics = plate_model.val(data=str(PLATE_DATA), imgsz=PLATE_IMGSZ, split="test", verbose=False,
                          project=str(WORK / "runs"), name="plate_test", exist_ok=True)
summary = (f"Test set  precision {metrics.box.mp:.3f}  recall {metrics.box.mr:.3f}  "
           f"mAP50 {metrics.box.map50:.3f}  mAP50-95 {metrics.box.map:.3f}")
print(summary)
(Path(OUTPUT_DIR) / "plate_detector_metrics.txt").write_text(summary + "\n")

test_images = sorted((PLATE_DATA.parent / "test/images").iterdir())[:6]
fig, axes = plt.subplots(1, len(test_images), figsize=(4 * len(test_images), 4), squeeze=False)
for ax, path in zip(axes[0], test_images):
    ax.imshow(plate_model(path, imgsz=PLATE_IMGSZ, verbose=False)[0].plot()[..., ::-1]); ax.axis("off")
fig.suptitle("Plate finder on unseen test images"); plt.show()

## 7. Export for the phone

This converts the model to TFLite (float16 weights, about 6 MB) and checks that it has the input and output shapes the app's `YoloDetector` expects. It then runs the TFLite file with the **same letterbox and decoding as the app** and draws the result, as a last check before copying it to the phone.

In [ ]:
run("export_tflite.py", "--weights", PLATE_WEIGHTS, "--name", "plate_detector",
    "--imgsz", PLATE_IMGSZ, "--assets", OUTPUT_DIR)

In [ ]:
import numpy as np
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    from tensorflow.lite.python.interpreter import Interpreter

def app_detect(tflite_path, labels, image, min_score=0.35):
    # Python port of yolo_detector.dart: letterbox, run, decode, NMS.
    it = Interpreter(str(tflite_path)); it.allocate_tensors()
    size = it.get_input_details()[0]["shape"][1]
    W, H = image.size; scale = min(size / W, size / H)
    w, h = max(1, round(W * scale)), max(1, round(H * scale))
    pad_x, pad_y = (size - w) // 2, (size - h) // 2
    x = np.full((size, size, 3), 114 / 255, np.float32)
    x[pad_y:pad_y + h, pad_x:pad_x + w] = np.asarray(image.resize((w, h), Image.BILINEAR), np.float32) / 255
    it.set_tensor(it.get_input_details()[0]["index"], x[None]); it.invoke()
    out = it.get_tensor(it.get_output_details()[0]["index"])[0]
    coord = size if out[2].max() <= 1.5 else 1
    found = []
    for a in range(out.shape[1]):
        c = int(out[4:, a].argmax()); s = float(out[4 + c, a])
        if s <= min_score: continue
        cx, cy, bw, bh = out[:4, a] * coord
        found.append((labels[c], s, ((cx - bw / 2 - pad_x) / scale, (cy - bh / 2 - pad_y) / scale,
                                     (cx + bw / 2 - pad_x) / scale, (cy + bh / 2 - pad_y) / scale)))
    found.sort(key=lambda f: -f[1]); kept = []
    def iou(a, b):
        ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
        i = ix * iy; return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i + 1e-9)
    for f in found:
        if not any(k[0] == f[0] and iou(k[2], f[2]) > 0.45 for k in kept): kept.append(f)
    return kept

labels = (Path(OUTPUT_DIR) / "plate_detector.txt").read_text().split()
im = Image.open(test_images[0]).convert("RGB")
hits = app_detect(Path(OUTPUT_DIR) / "plate_detector.tflite", labels, im)
d = ImageDraw.Draw(im)
for label, score, box in hits[:3]:
    d.rectangle(box, outline=(0, 200, 90), width=4); print(label, round(score, 2), [round(v) for v in box])
plt.imshow(im); plt.axis("off"); plt.title("TFLite, decoded like the app"); plt.show()

## 8. Vehicle type model

**8a. COCO vehicle detector (no training):** this exports the ready-made YOLOv8n. It knows car, motorcycle, bus and truck. The app treats a COCO "car" as possibly a van or a three-wheeler, so it never raises a false alarm on those.

**8b. Sri Lanka vehicle types with three-wheelers (optional):** this trains `vehicle_detector_lk.tflite`. When it's present, the app uses it in place of the COCO model, so a tuk-tuk is recognised as a three-wheeler instead of a "car" or "motorcycle".

The recommended data is the Kaggle dataset [**Vehicle Dataset for YOLO**](https://www.kaggle.com/datasets/nadinpethiyagoda/vehicle-dataset-for-yolo) by Nadin Pethiyagoda (licence DbCL v1.0). It has 3,000 images with exactly the app's six types: Car, Van, Bus, Truck (lorry), Motorbike and Three-wheel. To use it:

1. Choose **+ Add Input**, search "Vehicle Dataset for YOLO", and add it.
2. Set `TRAIN_THREE_WHEELER_MODEL = True` in section 1, then re-run from section 1.

Class names are mapped automatically ("Three-wheel" → `three_wheeler`, "Truck" → `lorry`), and the cell prints the mapping so you can check it. Section 3 never uses this dataset for plates.

To add your own photos, label **every vehicle in every photo** with the six types. If you label only the tuk-tuks, the model forgets the other types.

In [ ]:
if EXPORT_COCO_VEHICLE_MODEL:
    run("export_tflite.py", "--weights", BASE_WEIGHTS, "--name", "vehicle_detector",
        "--imgsz", 320, "--assets", OUTPUT_DIR)
    import ultralytics
    bus = Image.open(Path(ultralytics.__file__).parent / "assets/bus.jpg").convert("RGB")
    vlabels = (Path(OUTPUT_DIR) / "vehicle_detector.txt").read_text().splitlines()
    for label, score, box in app_detect(Path(OUTPUT_DIR) / "vehicle_detector.tflite", vlabels, bus)[:4]:
        print(label, round(score, 2), [round(v) for v in box])

In [ ]:
if TRAIN_THREE_WHEELER_MODEL:
    vehicle_source = Path(VEHICLE_DATA_FOLDER).resolve() if VEHICLE_DATA_FOLDER else None
    if vehicle_source is None and ON_KAGGLE:
        hits = [p for p in dataset_roots() if VEHICLE_KAGGLE_DATASET in str(p)]
        vehicle_source = hits[0] if hits else None
    assert vehicle_source, ("Add the 'Vehicle Dataset for YOLO' Kaggle input "
                            "(+ Add Input) or set VEHICLE_DATA_FOLDER in section 1")
    print("Vehicle dataset:", vehicle_source)

    run("prepare_dataset.py", "--task", "vehicles", "--source", vehicle_source,
        "--out", WORK / "datasets/vehicles")
    VEHICLE_DATA = WORK / "datasets/vehicles/data.yaml"
    show_labelled(VEHICLE_DATA.parent / "train", title="Vehicle training images (all classes shown in pink)")

    # Vehicles can be mirrored, unlike plate text: horizontal flips on.
    run("train.py", "--data", VEHICLE_DATA, "--name", "vehicle_detector_lk",
        "--imgsz", 320, "--epochs", VEHICLE_EPOCHS, "--batch", BATCH, "--device", DEVICE,
        "--model", BASE_WEIGHTS, "--fliplr", 0.5)
    LK_WEIGHTS = WORK / "runs/vehicle_detector_lk/weights/best.pt"

    # Per-class test scores: three_wheeler is the one that matters most.
    lk = YOLO(LK_WEIGHTS)
    m = lk.val(data=str(VEHICLE_DATA), imgsz=320, split="test", verbose=False,
               project=str(WORK / "runs"), name="vehicle_test", exist_ok=True)
    lines = [f"Test mAP50 {m.box.map50:.3f}  mAP50-95 {m.box.map:.3f}"]
    for i, c in enumerate(m.box.ap_class_index):
        lines.append(f"  {lk.names[int(c)]:14} mAP50 {m.box.ap50[i]:.3f}")
    print("\n".join(lines))
    (Path(OUTPUT_DIR) / "vehicle_detector_lk_metrics.txt").write_text("\n".join(lines) + "\n")
    shutil.copy(LK_WEIGHTS, Path(OUTPUT_DIR) / "vehicle_detector_lk_best.pt")
    if (LK_WEIGHTS.parent.parent / "results.png").exists():
        shutil.copy(LK_WEIGHTS.parent.parent / "results.png", Path(OUTPUT_DIR) / "vehicle_lk_results.png")

    run("export_tflite.py", "--weights", LK_WEIGHTS, "--name", "vehicle_detector_lk",
        "--imgsz", 320, "--assets", OUTPUT_DIR)
    test_imgs = sorted((VEHICLE_DATA.parent / "test/images").iterdir())[:4]
    lk_labels = (Path(OUTPUT_DIR) / "vehicle_detector_lk.txt").read_text().splitlines()
    for path in test_imgs:
        hits = app_detect(Path(OUTPUT_DIR) / "vehicle_detector_lk.tflite", lk_labels,
                          Image.open(path).convert("RGB"))
        print(path.name, [(label, round(score, 2)) for label, score, _ in hits[:3]])
else:
    print("Skipped (optional): no vehicle_detector_lk.tflite this run. The app uses "
          "vehicle_detector.tflite instead. To train it, add the 'Vehicle Dataset for "
          "YOLO' Kaggle input and set TRAIN_THREE_WHEELER_MODEL = True (see above).")

## 9. Before and after, for the report (optional)

Upload 50–100 test photos of real plates as a Kaggle dataset and add it as an input. They must not be in the training data: include some in the day, some at night and some at an angle. Put a CSV next to them:

```
image,plate,vehicle_type,colour,condition
IMG_0001.jpg,CAB1234,car,white,day
IMG_0002.jpg,QL4567,three_wheeler,green,night
```

Set `EVAL_IMAGES`, `EVAL_CSV` and `RUN_EVALUATION = True` in section 1. The report compares **before** (today's app: OCR on the scan-frame crop) with **after** (OCR on the plate finder's crop). It covers plate found, plate read, type right, colour right and colour "not sure". EasyOCR stands in for ML Kit, which only runs on the phone, so compare before and after with each other rather than treating the numbers as the phone's exact accuracy.

`evaluate.py` imports `plate_ocr/src/vehicle_colour.py`, so attach the whole `plate_ocr` folder for this step, not just `training/`.

In [ ]:
if RUN_EVALUATION:
    pip("easyocr")
    run("evaluate.py", "--images", Path(EVAL_IMAGES).resolve(), "--csv", Path(EVAL_CSV).resolve(),
        "--plate-weights", PLATE_WEIGHTS, "--ocr", "easyocr", "--out", Path(OUTPUT_DIR) / "evaluation.csv")
else:
    print("Skipped: set RUN_EVALUATION = True with EVAL_IMAGES and EVAL_CSV.")

## 10. Put the models in the app

The last cell zips `amica_models/` and shows a download link. Clicking **Save Version** also keeps everything in the notebook's **Output** tab. Then:

1. Copy the `.tflite` and `.txt` files into `amica-mobile-app/assets/models/`. The `.pt`, `.png` and metrics files are for the report and aren't needed in the app.
2. In `amica-mobile-app/lib/core/constants/feature_flags.dart`:
   * set `plateFinder = true` (plate finder)
   * set `vehicleCheck = true` (type and colour check, "What was I told?", match card). Deploy the backend first: `firebase deploy --only firestore:rules,functions`.
3. Run `flutter pub get`, then rebuild the app.

In [ ]:
from IPython.display import FileLink, display
print(sorted(p.name for p in Path(OUTPUT_DIR).iterdir()))
archive = shutil.make_archive(str(Path(OUTPUT_DIR).parent / "amica_models"), "zip", OUTPUT_DIR)
print("Packed:", archive)
if ON_KAGGLE:
    os.chdir("/kaggle/working")
    display(FileLink(os.path.relpath(archive, "/kaggle/working")))